#1. Inequality Constraints
In this notebook, we will learn how to handle inequality constraints in quantum annealing. Specifically, using the "Capacity-Constrained Knapsack Problem" as an example, we will implement the process of converting mathematical expressions (Hamiltonians) into Python code (QUBO matrices) step-by-step.

## 2. Problem Definition

### 2.1. What is the Knapsack Problem?

* There are $N$ items.
* Each item $i$ has an assigned weight $w_i$ and value $v_i$.
* The capacity of the knapsack is $C$.  
**Objective:** To maximize the total value of the items placed in the knapsack such that their total weight does not exceed the capacity $C$.

### 2.2. Standard Formulation

Mathematically, this problem is described as follows:

$$
\begin{aligned}
& \text{Maximize} & & \sum_{i=0}^{N-1} v_i x_i \\
& \text{subject to} & & \sum_{i=0}^{N-1} w_i x_i \le C \\
& & & x_i \in \{0, 1\}
\end{aligned}
$$

### 2.3. Explanation of Variables

The meanings of the symbols appearing in this formula are as follows:

| Symbol | Name | Description |
| :--- | :--- | :--- |
| **$N$** | Number of Items | The total number of items available. |
| **$i$** | Index | The item number ($0, 1, \dots, N-1$). |
| **$x_i$** | **Decision Variable** | The variable we want to determine through optimization.<br>・$1$: **Include** item $i$ in the knapsack<br>・$0$: **Do not include** item $i$ in the knapsack |
| **$v_i$** | Value | The value of item $i$ (constant). We want to maximize the sum of these values. |
| **$w_i$** | Weight | The weight of item $i$ (constant). |
| **$C$** | Capacity | The maximum weight the knapsack can hold (constant). |

## 3. Mathematical Model Construction (Converting Inequality to Equality)

### 3.1. Objective Function
"Maximizing value" is equivalent to "minimizing negative value".

$$H_{obj} = - \sum_{i=0}^{N-1} v_i x_i$$

### 3.2. Constraints (Handling Inequalities)
The inequality constraint "total weight $\le C$" is difficult to handle directly. Therefore, we introduce auxiliary variables $y_k$ representing that "the actual weight placed in the knapsack is $k$ kg" to convert it into an equality constraint.

Here, $k$ takes values of $1, 2, \dots, C$, and **exactly one of the auxiliary variables $y_1, y_2, \dots, y_C$ must be 1 (One-hot)**.

The constraints are expressed as the following two penalty terms:

1.  **One-hot Constraint ($H_{cost}^{(1)}$):** Exactly one of the auxiliary variables $y_k$ must be 1.
$$H_{cost}^{(1)} = \lambda \left( 1 - \sum_{k=1}^{C} y_k\right)^2$$
2.  **Weight Matching Constraint ($H_{cost}^{(2)}$):** The weight $k$ indicated by the selected auxiliary variable matches the actual total weight of the items.
$$H_{cost}^{(2)} = \lambda \left( \sum_{k=1}^{C} k y_k - \sum_{i=0}^{N-1} w_i x_i \right)^2$$

($\lambda$ is a large positive constant representing the strength of the penalty.)

### 3.3. Total Hamiltonian

The sum of these is the energy function that we ultimately need to minimize.

$$H = H_{obj} + H_{cost}^{(1)} + H_{cost}^{(2)}$$
$$H = - \sum_{i} v_i x_i + \lambda \left( 1 - \sum_{k} y_k \right)^2 + \lambda \left( \sum_{k} k y_k - \sum_{i} w_i x_i \right)^2$$

## 4. Transformation of the Hamiltonian into QUBO Format

To implement this in a program, we need to expand the Hamiltonian into a form consisting of products of variables (QUBO format), such as $x_i x_j$ and $y_k y_l$.
We expand this utilizing the property of binary variables, $x^2 = x$.

### 4.1. Objective Function $H_{obj}$

Since this is already in a simple form, it directly becomes the diagonal elements $-v_i$.
$$H_{obj} = \sum_{i} (-v_i) x_i$$

### 4.2. Constraint Term 1 $H_{cost}^{(1)}$ (One-hot Constraint)

Equation: $\lambda ( 1 - \sum_{k} y_k )^2$
\begin{aligned}
H_{cost}^{(1)} &= \lambda \left( 1 - 2\sum_{k} y_k + \left(\sum_{k} y_k\right)^2 \right) \\
&= \lambda \left( 1 - 2\sum_{k} y_k + \left( \sum_{k} y_k^2 + \sum_{k \neq l} y_k y_l \right) \right) \\
&\quad \text{* Applying the binary variable property } y_k^2 = y_k \\
&= \lambda \left( 1 - 2\sum_{k} y_k + \sum_{k} y_k + 2\sum_{k < l} y_k y_l \right) \\
&= \lambda \left( 1 - \sum_{k} y_k + 2\sum_{k < l} y_k y_l \right) \\
&\simeq \sum_{k} (-\lambda) y_k + \sum_{k < l} (2\lambda) y_k y_l \quad \text{(Ignoring the constant term)}
\end{aligned}

### 4.3. Constraint Term 2 $H_{cost}^{(2)}$ (Weight Matching Constraint)

This is the most complex part. We will divide the equation into three parts and expand them.
$$H_{cost}^{(2)} = \lambda \left( \underbrace{\sum_{k=1}^{C} k y_k}_{A} - \underbrace{\sum_{i=0}^{N-1} w_i x_i}_{B} \right)^2 = \lambda(A^2 - 2AB + B^2)$$

#### 4.3.1.**Part A: Squared term of auxiliary variables ($A^2$)**
$$
\begin{aligned}
\lambda \left(\sum_{k} k y_k\right)^2 &= \lambda \left( \sum_{k} k^2 y_k + 2\sum_{k < l} k l y_k y_l \right)
\end{aligned}
$$
Mapping to matrix $Q$:
* **$Q_{yy}$ (Diagonal):** $\lambda k^2$
* **$Q_{yl}$ (Cross term $k<l$):** $2 \lambda k l$

#### 4.3.2.**Part B: Squared term of item variables ($B^2$)**
$$
\begin{aligned}
\lambda \left(\sum_{i} w_i x_i\right)^2 &= \lambda \left( \sum_{i} w_i^2 x_i + 2\sum_{i < j} w_i w_j x_i x_j \right)
\end{aligned}
$$
Mapping to matrix $Q$:
* **$Q_{ii}$ (Diagonal):** $\lambda w_i^2$
* **$Q_{ij}$ (Cross term $i<j$):** $2 \lambda w_i w_j$

#### 4.3.3.**Part C: Cross term ($-2AB$)**
$$
\begin{aligned}
-2\lambda \left(\sum_{k} k y_k\right) \left(\sum_{i} w_i x_i\right) &= -2\lambda \sum_{k}\sum_{i} k w_i y_k x_i
\end{aligned}
$$
Mapping to matrix $Q$:
* **$Q_{yi}$ (Cross term between auxiliary and item variables):** $-2 \lambda k w_i$

## 5. Python Implementation and Solution with OpenJij

### 5.1. Installing the Required Libraries

In [1]:
pip install openjij numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 15.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


### 5.2. Importing Libraries

In [2]:
import numpy as np
import openjij as oj

### 5.3. Problem **Setup**

In [4]:
# ==========================================
# Problem Setup
# ==========================================
# Define the list of items.
# Each dictionary represents one item. The keys mean the following:
#   'weight': w_i in the formula (weight)
#   'value':  v_i in the formula (value)
items = [
    {'weight': 2, 'value': 3},
    {'weight': 3, 'value': 4},
    {'weight': 4, 'value': 5},
    {'weight': 5, 'value': 8}, # Heavy (5kg) but high value (8). Choosing this is the key.
    {'weight': 1, 'value': 1}
]

# For easier calculation, separate weights and values into distinct lists.
# This allows accessing the i-th weight via weights[i] and the i-th value via values[i].
weights = [item['weight'] for item in items]
values = [item['value'] for item in items]

# ==========================================
# Definition of Constants and Parameters
# ==========================================
N = len(items)       # Total number of items (N in the formula)
                     # -> This means item variables x_0 ... x_{N-1} are required.

C = 7                # Capacity limit of the knapsack (C in the formula)
                     # -> Total weight must be less than or equal to this (Σw_i x_i <= C).

# Penalty coefficient (λ: Lambda in the formula)
# The strength of the penalty imposed when constraints (capacity overflow or One-hot violation) are broken.
# If this value is too small, the solver might decide "it's more profitable to include high-value items even with the penalty,"
# resulting in a solution that violates the rules.
# Therefore, we set a value sufficiently larger than the item values (here, 100.0).
LAMBDA = 100.0

# Display the setup for confirmation
print(f"Items (Weight, Value): {list(zip(weights, values))}")
print(f"Capacity: {C}")

Items (Weight, Value): [(2, 3), (3, 4), (4, 5), (5, 8), (1, 1)]
Capacity: 7


### 5.4. Creating the QUBO Matrix

In [7]:
# ==========================================
# QUBO Matrix Initialization (Zero Matrix)
# ==========================================
# Create a blank (N+C) x (N+C) matrix
Q = np.zeros((N+C, N+C))

# --- Utility functions for index manipulation ---
def get_q_idx(i):
    """Index in the matrix for item i (0 to N-1)"""
    return i

def get_y_idx(k):
    """Index in the matrix for auxiliary variable y_k (k=1 to C)"""
    # Follows the item variables. Since k starts at 1, subtract 1 to make it 0-indexed.
    return N + (k - 1)

# ==========================================
# Embedding into the QUBO Matrix
# ==========================================

# ------------------------------------------------
# (A) Objective Function: Maximize value (Minimize negative value)
# ------------------------------------------------
for i in range(N):
    idx = get_q_idx(i)
    v = items[i]['value']
    # Add -v to the diagonal component
    Q[idx, idx] += -1 * v

# ------------------------------------------------
# (B) Constraint 1: Exactly one auxiliary variable y must be 1 (One-hot)
# Formula: lambda * (1 - sum(y))^2
# Expansion -> lambda * ( sum(y^2) + sum(y_i*y_j) - 2sum(y) ) + constant
# ------------------------------------------------
for k1 in range(1, C + 1):
    idx1 = get_y_idx(k1)

    # 1. Linear term: -2 * lambda * y_k
    Q[idx1, idx1] += -2 * LAMBDA

    # 2. Squared term diagonal component: + lambda * y_k^2 (= y_k)
    Q[idx1, idx1] += LAMBDA

    # 3. Squared term cross component: + 2 * lambda * y_k1 * y_k2
    for k2 in range(k1 + 1, C + 1):
        idx2 = get_y_idx(k2)
        Q[idx1, idx2] += 2 * LAMBDA

# ------------------------------------------------
# (C) Constraint 2: Match the total weights
# Formula: lambda * ( sum(k*y) - sum(w*q) )^2
# Expansion -> lambda * ( A^2 - 2AB + B^2 )
# A = sum(k*y), B = sum(w*q)
# ------------------------------------------------

# --- Part 1: A^2 (Product of auxiliary variables) ---
# (sum k*y)^2 = sum k^2*y + sum 2*k1*k2*y1*y2
for k1 in range(1, C + 1):
    idx1 = get_y_idx(k1)
    # Diagonal component (k^2)
    Q[idx1, idx1] += LAMBDA * (k1 ** 2)

    # Cross component (2 * k1 * k2)
    for k2 in range(k1 + 1, C + 1):
        idx2 = get_y_idx(k2)
        Q[idx1, idx2] += LAMBDA * 2 * k1 * k2

# --- Part 2: B^2 (Product of item variables) ---
# (sum w*q)^2 = sum w^2*q + sum 2*w1*w2*q1*q2
for i1 in range(N):
    idx1 = get_q_idx(i1)
    w1 = items[i1]['weight']
    # Diagonal component (w^2)
    Q[idx1, idx1] += LAMBDA * (w1 ** 2)

    # Cross component (2 * w1 * w2)
    for i2 in range(i1 + 1, N):
        idx2 = get_q_idx(i2)
        w2 = items[i2]['weight']
        Q[idx1, idx2] += LAMBDA * 2 * w1 * w2

# --- Part 3: -2AB (Product of item and auxiliary variables) ---
# -2 * sum(k*y) * sum(w*q)
for k in range(1, C + 1):
    idx_y = get_y_idx(k)
    for i in range(N):
        idx_q = get_q_idx(i)
        w = items[i]['weight']

        # Coefficient: -2 * lambda * k * w
        coeff = -2 * LAMBDA * k * w

        # Process to place in the upper triangular part of the matrix (row < col)
        if idx_y < idx_q:
            Q[idx_y, idx_q] += coeff
        else:
            Q[idx_q, idx_y] += coeff

### 5.5. Solving with OpenJij

In [12]:
# ==========================================
# Solving with OpenJij
# ==========================================
print("Calculating optimization...")
sampler = oj.SQASampler()
# num_reads: Number of calculations (higher means more stable)
response = sampler.sample_qubo(Q, num_reads=1000)

# Getting the best solution
sample = response.first.sample

# ==========================================
# Displaying the results
# ==========================================
print("\n" + "="*30)
print(" Optimization Results ")
print("="*30)

total_w = 0
total_v = 0
active_y = 0

# --- Checking the items ---
print("[Selected Items]")
for i in range(N):
    idx = get_q_idx(i)
    if sample[idx] == 1:
        item = items[i]
        print(f"  ID:{i} (Weight:{item['weight']}, Value:{item['value']})")
        total_w += item['weight']
        total_v += item['value']

# --- Checking the auxiliary variables ---
for k in range(1, C + 1):
    idx = get_y_idx(k)
    if sample[idx] == 1:
        active_y = k

print("-" * 30)
print(f"Total Value: {total_v}")
print(f"Total Weight: {total_w} / Capacity {C}")
print(f"Auxiliary Variable (Target Weight): {active_y}")

if total_w <= C and total_w == active_y:
    print(">>> Success: Optimal solution satisfying the constraints.")
else:
    print(">>> Failure: Constraint violation occurred (possible lack of penalty).")

Calculating optimization...

 Optimization Results 
[Selected Items]
  ID:0 (Weight:2, Value:3)
  ID:3 (Weight:5, Value:8)
------------------------------
Total Value: 11
Total Weight: 7 / Capacity 7
Auxiliary Variable (Target Weight): 7
>>> Success: Optimal solution satisfying the constraints.


### 5.6. Validity of the Solution

First, let's verify if the constraints are satisfied.
* **Selected items:** ID:0 (2kg) and ID:3 (5kg)
* **Total weight:** 2 + 5 = 7 kg
* **Capacity limit:** $C$ = 7 kg

The total weight does not exceed the capacity limit ($\le$ 7).
Furthermore, please pay attention to the "Auxiliary Variable (Target Weight): 7" in the output. This means the annealing machine determined that "the total weight of the items will be 7kg" and correctly set the corresponding slack variable $y_7$ to 1.
The weight of the items and the auxiliary variable are perfectly synchronized, which is evidence that the mathematical model ($H_{cost}^{(2)}$) functioned correctly.

### 5.7. Confirmation of Optimality

Is this really the best combination (the optimal solution)? Let's compare it with other combinations.

* **Pattern A (Current solution):**
    * Item 0 (2kg, Value 3) + Item 3 (5kg, Value 8)
    * $\to$ Weight 7kg, **Value 11**
* **Pattern B (Another combination example):**
    * Item 1 (3kg, Value 4) + Item 2 (4kg, Value 5)
    * $\to$ Weight 7kg, **Value 9**
* **Pattern C (Many light items):**
    * Item 0 + Item 1 + Item 4
    * $\to$ Weight 6kg, **Value 8**

The annealing machine has correctly searched for the combination that maximizes the value, rather than just satisfying the weight constraint.